# MNQ ORB - Comparatif `sizing_3state` vs ORB retenue

Ce notebook compare les deux versions ORB qui ont servi a deux lectures differentes du repo:

- `sizing_3state`: la version la plus poussee pour la lecture prop / survivabilite.
- `retained final`: la version ORB retenue dans la campagne de recherche generale.

Le but ici est simple: remettre les deux sur des graphiques comparables, avec le meme esprit de charting que le notebook d'ensemble.


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"


def fmt_pct_from_ratio(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value) * 100.0:.1f}%"


def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"


def build_curve_from_daily(daily, initial_balance):
    out = daily.copy()
    out["session_date"] = pd.to_datetime(out["session_date"])
    out = out.sort_values("session_date").reset_index(drop=True)
    out["daily_pnl_usd"] = pd.to_numeric(out["daily_pnl_usd"], errors="coerce").fillna(0.0)
    out["equity"] = initial_balance + out["daily_pnl_usd"].cumsum()
    out["peak_equity"] = out["equity"].cummax()
    out["drawdown_usd"] = out["equity"] - out["peak_equity"]
    out["drawdown_pct"] = (out["equity"] / out["peak_equity"] - 1.0) * 100.0
    out["timestamp"] = out["session_date"]
    return out


def build_curve_from_equity_points(curve_df):
    out = curve_df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce").dt.tz_convert(None)
    out = out.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    out["equity"] = pd.to_numeric(out["equity"], errors="coerce")
    out = out.dropna(subset=["equity"]).reset_index(drop=True)
    out["peak_equity"] = out["equity"].cummax()
    out["drawdown_usd"] = out["equity"] - out["peak_equity"]
    out["drawdown_pct"] = (out["equity"] / out["peak_equity"] - 1.0) * 100.0
    out["session_date"] = out["timestamp"].dt.normalize()
    return out


def rebase_curve_segment(curve_df, start_timestamp, initial_balance):
    curve = curve_df.sort_values("timestamp").reset_index(drop=True).copy()
    start_ts = pd.Timestamp(start_timestamp)
    curve = curve.loc[curve["timestamp"] >= start_ts].copy()
    if curve.empty:
        raise ValueError("No rows available after requested start timestamp.")

    previous_rows = curve_df.loc[curve_df["timestamp"] < start_ts].sort_values("timestamp")
    prior_equity = float(previous_rows["equity"].iloc[-1]) if not previous_rows.empty else initial_balance

    curve["equity"] = initial_balance + (curve["equity"] - prior_equity)
    curve["peak_equity"] = curve["equity"].cummax()
    curve["drawdown_usd"] = curve["equity"] - curve["peak_equity"]
    curve["drawdown_pct"] = (curve["equity"] / curve["peak_equity"] - 1.0) * 100.0
    curve["session_date"] = curve["timestamp"].dt.normalize()
    return curve.reset_index(drop=True)


def curve_to_daily_pnl(curve_df):
    curve = curve_df.sort_values("timestamp").reset_index(drop=True).copy()
    curve["prev_equity"] = curve["equity"].shift(1)
    curve["pnl_increment"] = curve["equity"] - curve["prev_equity"]
    curve.loc[curve["prev_equity"].isna(), "pnl_increment"] = curve.loc[curve["prev_equity"].isna(), "equity"] - 50000.0
    curve["session_date"] = curve["timestamp"].dt.normalize()
    daily = curve.groupby("session_date", as_index=False)["pnl_increment"].sum().rename(columns={"pnl_increment": "daily_pnl_usd"})
    daily["session_date"] = pd.to_datetime(daily["session_date"])
    return daily


In [2]:
REGIME_EXPORT_ROOT = ROOT / r"data\exports\mnq_orb_regime_filter_sizing_20260325_150405"
PROP_EXPORT_ROOT = ROOT / r"data\exports\mnq_orb_prop_challenge_readiness_20260328_run"
RESEARCH_EXPORT_ROOT = ROOT / r"export\orb_research_campaign"
VARIANT_NAME = "sizing_3state_realized_vol_ratio_15_60"
RETAINED_CONFIG_NAME = "full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate"
INITIAL_BALANCE_USD = 50_000.0

required_paths = {
    "regime_export_root": REGIME_EXPORT_ROOT,
    "prop_export_root": PROP_EXPORT_ROOT,
    "research_export_root": RESEARCH_EXPORT_ROOT,
    "summary_variants": REGIME_EXPORT_ROOT / "summary_variants.csv",
    "variant_metrics": REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "metrics_by_scope.csv",
    "variant_daily": REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "daily_results.csv",
    "variant_controls": REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "controls.csv",
    "regime_mapping": REGIME_EXPORT_ROOT / "regime_state_mappings.csv",
    "retained_results": RESEARCH_EXPORT_ROOT / "full_reopt_results.csv",
    "retained_curve": RESEARCH_EXPORT_ROOT / "charts" / "equity_curve__full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate.csv",
    "campaign_report": RESEARCH_EXPORT_ROOT / "campaign_report.md",
    "prop_final_report": PROP_EXPORT_ROOT / "final_report.md",
    "prop_final_verdict": PROP_EXPORT_ROOT / "final_verdict.json",
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Fichiers manquants pour le notebook: {missing}")

print("REGIME_EXPORT_ROOT =", REGIME_EXPORT_ROOT)
print("PROP_EXPORT_ROOT   =", PROP_EXPORT_ROOT)
print("RESEARCH_EXPORT    =", RESEARCH_EXPORT_ROOT)
print("VARIANT_NAME       =", VARIANT_NAME)
print("RETAINED_CONFIG    =", RETAINED_CONFIG_NAME)


REGIME_EXPORT_ROOT = C:\Data\Perso\algo-trading-intraday-research\data\exports\mnq_orb_regime_filter_sizing_20260325_150405
PROP_EXPORT_ROOT   = C:\Data\Perso\algo-trading-intraday-research\data\exports\mnq_orb_prop_challenge_readiness_20260328_run
RESEARCH_EXPORT    = C:\Data\Perso\algo-trading-intraday-research\export\orb_research_campaign
VARIANT_NAME       = sizing_3state_realized_vol_ratio_15_60
RETAINED_CONFIG    = full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate


In [3]:
regime_metadata = json.loads((REGIME_EXPORT_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
prop_verdict = json.loads((PROP_EXPORT_ROOT / "final_verdict.json").read_text(encoding="utf-8"))

summary_variants = pd.read_csv(REGIME_EXPORT_ROOT / "summary_variants.csv")
variant_metrics = pd.read_csv(REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "metrics_by_scope.csv")
variant_daily = pd.read_csv(REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "daily_results.csv", parse_dates=["session_date"])
variant_controls = pd.read_csv(REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "controls.csv", parse_dates=["session_date"])
regime_mapping = pd.read_csv(REGIME_EXPORT_ROOT / "regime_state_mappings.csv")

retained_results = pd.read_csv(RESEARCH_EXPORT_ROOT / "full_reopt_results.csv")
retained_curve_raw = pd.read_csv(RESEARCH_EXPORT_ROOT / "charts" / "equity_curve__full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate.csv")

variant_row = summary_variants.loc[summary_variants["variant_name"] == VARIANT_NAME].iloc[0]
oos_start_date = pd.to_datetime(variant_controls.loc[variant_controls["phase"] == "oos", "session_date"].min())

retained_row = retained_results.loc[
    (retained_results["name"] == RETAINED_CONFIG_NAME)
    & (retained_results["compression_mode"] == "weak_close")
    & (retained_results["dynamic_mode"] == "noise_area_gate")
    & (retained_results["exit_mode"] == "baseline")
    & (retained_results["noise_lookback"] == 30)
].iloc[0]

bucket_map = (
    regime_mapping.loc[
        (regime_mapping["variant_name"] == VARIANT_NAME)
        & (regime_mapping["feature_name"] == "realized_vol_ratio_15_60"),
        ["bucket_label", "lower_bound", "upper_bound", "risk_multiplier", "oos_n_obs", "oos_net_pnl", "oos_sharpe", "oos_max_drawdown"],
    ]
    .drop_duplicates()
    .sort_values(["risk_multiplier", "bucket_label"])
    .reset_index(drop=True)
)
bucket_map["effective_risk_per_trade_pct"] = bucket_map["risk_multiplier"] * float(regime_metadata["spec"]["baseline"]["risk_per_trade_pct"])

variant_curve = build_curve_from_daily(variant_daily, INITIAL_BALANCE_USD)
variant_curve_oos = build_curve_from_daily(variant_daily.loc[variant_daily["session_date"] >= oos_start_date], INITIAL_BALANCE_USD)

retained_curve = build_curve_from_equity_points(retained_curve_raw)
retained_curve_oos = rebase_curve_segment(retained_curve, oos_start_date, INITIAL_BALANCE_USD)

variant_daily_oos = variant_daily.loc[variant_daily["session_date"] >= oos_start_date, ["session_date", "daily_pnl_usd"]].copy()
retained_daily_full = curve_to_daily_pnl(retained_curve)
retained_daily_oos = retained_daily_full.loc[retained_daily_full["session_date"] >= oos_start_date].copy()

retained_config = json.loads(retained_row["config_json"])
display(Markdown(f"**OOS start date shared for the comparison:** `{oos_start_date.date()}`"))


**OOS start date shared for the comparison:** `2024-02-23`

## 1. Lecture Rapide

Avant de lire les courbes, il faut garder la logique de decision en tete:

- `sizing_3state` = overlay de risque / survivabilite sur une base ORB nominale.
- `retained final` = config ORB finalement retenue dans la campagne de recherche generale.
- Les deux ne repondent donc pas exactement au meme objectif, meme si la comparaison visuelle reste utile.


In [4]:
summary_lines = [
    "### Resume net",
    f"- `sizing_3state` lit le repo sous un angle prop: OOS net `{fmt_money(variant_row['oos_net_pnl'])}`, Sharpe `{fmt_float(variant_row['oos_sharpe'])}`, maxDD `{fmt_money(variant_row['oos_max_drawdown'])}`.",
    f"- `retained final` est la config ORB retenue dans la campagne generale: OOS net `{fmt_money(retained_row['oos_net_pnl'])}`, Sharpe `{fmt_float(retained_row['oos_sharpe_ratio'])}`, maxDD `{fmt_money(retained_row['oos_max_drawdown'])}`.",
    f"- Verdict prop recharge: meilleure variante de challenge = `{prop_verdict['recommended_launch_variant']}` avec profil `{prop_verdict['recommended_launch_risk_profile']}`.",
    "- Lecture pratique: `sizing_3state` si tu privilegies la survivabilite prop, `retained final` si tu veux relire la config ORB officiellement retenue dans la recherche du repo.",
]

display(Markdown("\n".join(summary_lines)))


### Resume net
- `sizing_3state` lit le repo sous un angle prop: OOS net `28,826.0 USD`, Sharpe `2.109`, maxDD `-5,934.5 USD`.
- `retained final` est la config ORB retenue dans la campagne generale: OOS net `5,155.5 USD`, Sharpe `1.559`, maxDD `-590.5 USD`.
- Verdict prop recharge: meilleure variante de challenge = `baseline_3state` avec profil `assertive`.
- Lecture pratique: `sizing_3state` si tu privilegies la survivabilite prop, `retained final` si tu veux relire la config ORB officiellement retenue dans la recherche du repo.

## 2. Tableau Comparatif

On met les scopes `full / is / oos` cote a cote avec les memes colonnes simples.


In [5]:
variant_metrics_view = variant_metrics.rename(
    columns={
        "scope": "scope",
        "net_pnl": "net_pnl_usd",
        "sharpe": "sharpe",
        "profit_factor": "profit_factor",
        "max_drawdown": "max_drawdown_usd",
        "n_trades": "n_trades",
        "pct_days_traded": "pct_days_traded",
        "worst_day": "worst_day_usd",
    }
)[["scope", "net_pnl_usd", "sharpe", "profit_factor", "max_drawdown_usd", "n_trades", "pct_days_traded", "worst_day_usd"]].copy()
variant_metrics_view["strategy"] = "sizing_3state"

retained_metrics_view = pd.DataFrame(
    [
        {
            "strategy": "retained_final",
            "scope": "full",
            "net_pnl_usd": retained_row["overall_net_pnl"],
            "sharpe": retained_row["overall_sharpe_ratio"],
            "profit_factor": retained_row["overall_profit_factor"],
            "max_drawdown_usd": retained_row["overall_max_drawdown"],
            "n_trades": retained_row["overall_nb_trades"],
            "pct_days_traded": retained_row["overall_pct_days_traded"],
            "worst_day_usd": retained_row["overall_worst_day"],
        },
        {
            "strategy": "retained_final",
            "scope": "is",
            "net_pnl_usd": retained_row["is_net_pnl"],
            "sharpe": retained_row["is_sharpe_ratio"],
            "profit_factor": retained_row["is_profit_factor"],
            "max_drawdown_usd": retained_row["is_max_drawdown"],
            "n_trades": retained_row["is_nb_trades"],
            "pct_days_traded": retained_row["is_pct_days_traded"],
            "worst_day_usd": retained_row["is_worst_day"],
        },
        {
            "strategy": "retained_final",
            "scope": "oos",
            "net_pnl_usd": retained_row["oos_net_pnl"],
            "sharpe": retained_row["oos_sharpe_ratio"],
            "profit_factor": retained_row["oos_profit_factor"],
            "max_drawdown_usd": retained_row["oos_max_drawdown"],
            "n_trades": retained_row["oos_nb_trades"],
            "pct_days_traded": retained_row["oos_pct_days_traded"],
            "worst_day_usd": retained_row["oos_worst_day"],
        },
    ]
)

comparison_metrics = pd.concat([variant_metrics_view, retained_metrics_view], ignore_index=True)
comparison_metrics["net_pnl_usd"] = comparison_metrics["net_pnl_usd"].map(lambda v: round(float(v), 1))
comparison_metrics["sharpe"] = comparison_metrics["sharpe"].map(lambda v: round(float(v), 3))
comparison_metrics["profit_factor"] = comparison_metrics["profit_factor"].map(lambda v: round(float(v), 3))
comparison_metrics["max_drawdown_usd"] = comparison_metrics["max_drawdown_usd"].map(lambda v: round(float(v), 1))
comparison_metrics["pct_days_traded"] = comparison_metrics["pct_days_traded"].map(lambda v: round(float(v) * 100.0, 2))
comparison_metrics["worst_day_usd"] = comparison_metrics["worst_day_usd"].map(lambda v: round(float(v), 1))

display(comparison_metrics.sort_values(["scope", "strategy"]).reset_index(drop=True))


,scope,net_pnl_usd,sharpe,profit_factor,max_drawdown_usd,n_trades,pct_days_traded,worst_day_usd,strategy
0,full,9776.0,0.892,1.407,-1913.5,298,13.91,-249.0,retained_final
1,is,4620.5,0.604,1.249,-1913.5,222,14.80,-249.0,retained_final
2,is,11601.5,0.365,1.073,-8354.0,781,63.91,-750.0,sizing_3state
3,oos,5155.5,1.559,1.951,-590.5,76,11.82,-247.5,retained_final
4,oos,28826.0,2.109,1.537,-5934.5,326,62.10,-744.0,sizing_3state
5,overall,40427.5,0.889,1.191,-8354.0,1107,63.37,-750.0,sizing_3state


## 3. Equity / Drawdown

On reprend le meme genre de lecture que dans le notebook d'ensemble:

- courbe full sample,
- drawdown full sample,
- courbe OOS only,
- drawdown OOS only.


In [6]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Full sample equity",
        "Full sample drawdown %",
        "OOS only equity",
        "OOS only drawdown %",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

for name, curve, color in [
    ("Sizing 3-state full", variant_curve, "#16a34a"),
    ("Retained final full", retained_curve, "#2563eb"),
]:
    fig.add_trace(
        go.Scatter(x=curve["timestamp"], y=curve["equity"], mode="lines", name=name, line=dict(width=2, color=color)),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=curve["timestamp"], y=curve["drawdown_pct"], mode="lines", name=f"{name} DD", showlegend=False, line=dict(width=1.8, color=color, dash="dot")),
        row=1,
        col=2,
    )

for name, curve, color in [
    ("Sizing 3-state oos", variant_curve_oos, "#16a34a"),
    ("Retained final oos", retained_curve_oos, "#2563eb"),
]:
    fig.add_trace(
        go.Scatter(x=curve["timestamp"], y=curve["equity"], mode="lines", name=name, line=dict(width=2, color=color)),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=curve["timestamp"], y=curve["drawdown_pct"], mode="lines", name=f"{name} DD", showlegend=False, line=dict(width=1.8, color=color, dash="dot")),
        row=2,
        col=2,
    )

fig.update_yaxes(title_text="Equity (USD)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown %", row=1, col=2)
fig.update_yaxes(title_text="Equity (USD)", row=2, col=1)
fig.update_yaxes(title_text="Drawdown %", row=2, col=2)
fig.update_layout(
    height=900,
    width=1400,
    title="MNQ ORB comparison - sizing_3state vs retained final",
    legend=dict(orientation="h", y=1.08),
)
fig.show()


## 4. Parametrage

Ici on regarde rapidement ce que chaque version change vraiment.


In [7]:
display(Markdown("### Bucket map - sizing_3state"))
display(bucket_map)

retained_param_rows = [
    {"section": "entry", "parameter": "or_minutes", "value": retained_config["baseline_entry"]["or_minutes"]},
    {"section": "entry", "parameter": "direction", "value": retained_config["baseline_entry"]["direction"]},
    {"section": "entry", "parameter": "entry_buffer_ticks", "value": retained_config["baseline_entry"]["entry_buffer_ticks"]},
    {"section": "entry", "parameter": "stop_buffer_ticks", "value": retained_config["baseline_entry"]["stop_buffer_ticks"]},
    {"section": "entry", "parameter": "target_multiple", "value": retained_config["baseline_entry"]["target_multiple"]},
    {"section": "entry", "parameter": "vwap_confirmation", "value": retained_config["baseline_entry"]["vwap_confirmation"]},
    {"section": "ensemble", "parameter": "atr_window", "value": retained_config["baseline_ensemble"]["atr_window"]},
    {"section": "ensemble", "parameter": "vote_threshold", "value": retained_config["baseline_ensemble"]["vote_threshold"]},
    {"section": "overlay", "parameter": "compression_mode", "value": retained_config["compression"]["mode"]},
    {"section": "overlay", "parameter": "compression_usage", "value": retained_config["compression"]["usage"]},
    {"section": "overlay", "parameter": "dynamic_mode", "value": retained_config["dynamic_threshold"]["mode"]},
    {"section": "overlay", "parameter": "noise_lookback", "value": retained_config["dynamic_threshold"]["noise_lookback"]},
    {"section": "overlay", "parameter": "noise_vm", "value": retained_config["dynamic_threshold"]["noise_vm"]},
]

display(Markdown("### Param snapshot - retained final"))
display(pd.DataFrame(retained_param_rows))


### Bucket map - sizing_3state

,bucket_label,lower_bound,upper_bound,risk_multiplier,oos_n_obs,oos_net_pnl,oos_sharpe,oos_max_drawdown,effective_risk_per_trade_pct
0,low,0.336552,0.942945,0.50,141,11704.5,1.968231,-9419.5,0.750
1,high,1.140780,1.822058,0.75,100,1198.5,0.261698,-10840.0,1.125
2,mid,0.942945,1.140780,1.00,98,22672.0,5.276851,-2439.5,1.500


### Param snapshot - retained final

,section,parameter,value
0,entry,or_minutes,15
1,entry,direction,long
2,entry,entry_buffer_ticks,2
3,entry,stop_buffer_ticks,2
4,entry,target_multiple,2.0
5,entry,vwap_confirmation,True
6,ensemble,atr_window,14
7,ensemble,vote_threshold,0.5
8,overlay,compression_mode,weak_close
9,overlay,compression_usage,soft_vote_bonus


## 5. Distribution OOS des daily pnl

Ce chart donne une lecture rapide du profil de jour OOS sur les deux variantes.


In [8]:
oos_distribution = pd.concat(
    [
        variant_daily_oos.assign(strategy="sizing_3state"),
        retained_daily_oos.assign(strategy="retained_final"),
    ],
    ignore_index=True,
)
oos_distribution = oos_distribution.loc[oos_distribution["daily_pnl_usd"].fillna(0.0) != 0.0].copy()

fig = px.histogram(
    oos_distribution,
    x="daily_pnl_usd",
    color="strategy",
    nbins=60,
    barmode="overlay",
    opacity=0.55,
    title="Distribution OOS des daily pnl non nuls",
    labels={"daily_pnl_usd": "Daily pnl (USD)", "strategy": "Strategy"},
)
fig.show()


## 6. Conclusion

La derniere cellule reformule la difference de role entre les deux notebooks.


In [9]:
conclusion_lines = [
    "### Verdict simple",
    f"- `sizing_3state` est le plus fort ici si ton filtre principal est la survivabilite prop: OOS Sharpe `{fmt_float(variant_row['oos_sharpe'])}`, net `{fmt_money(variant_row['oos_net_pnl'])}`, maxDD `{fmt_money(variant_row['oos_max_drawdown'])}`.",
    f"- `retained final` est la version ORB officiellement retenue dans la recherche repo: OOS Sharpe `{fmt_float(retained_row['oos_sharpe_ratio'])}`, net `{fmt_money(retained_row['oos_net_pnl'])}`, maxDD `{fmt_money(retained_row['oos_max_drawdown'])}`.",
    "- Si tu veux une lecture client prop / challenge: repars du notebook `sizing_3state`.",
    "- Si tu veux la sleeve ORB finale du portefeuille recherche: repars du notebook `mnq_orb_retained_final`.",
]

display(Markdown("\n".join(conclusion_lines)))


### Verdict simple
- `sizing_3state` est le plus fort ici si ton filtre principal est la survivabilite prop: OOS Sharpe `2.109`, net `28,826.0 USD`, maxDD `-5,934.5 USD`.
- `retained final` est la version ORB officiellement retenue dans la recherche repo: OOS Sharpe `1.559`, net `5,155.5 USD`, maxDD `-590.5 USD`.
- Si tu veux une lecture client prop / challenge: repars du notebook `sizing_3state`.
- Si tu veux la sleeve ORB finale du portefeuille recherche: repars du notebook `mnq_orb_retained_final`.

## 7. Sources

Le notebook reste branche sur des exports et notebooks explicites.


In [10]:
source_paths = pd.DataFrame(
    [
        {"name": "regime_export_root", "path": str(REGIME_EXPORT_ROOT)},
        {"name": "prop_export_root", "path": str(PROP_EXPORT_ROOT)},
        {"name": "research_export_root", "path": str(RESEARCH_EXPORT_ROOT)},
        {"name": "sizing_notebook", "path": str(ROOT / "notebooks" / "orb_MNQ_sizing_3state_client.ipynb")},
        {"name": "retained_notebook", "path": str(ROOT / "notebooks" / "finals" / "mnq_orb_retained_final.ipynb")},
    ]
)
display(source_paths)


,name,path
0,regime_export_root,C:\Data\Perso\algo-trading-intraday-research\d...
1,prop_export_root,C:\Data\Perso\algo-trading-intraday-research\d...
2,research_export_root,C:\Data\Perso\algo-trading-intraday-research\e...
3,sizing_notebook,C:\Data\Perso\algo-trading-intraday-research\n...
4,retained_notebook,C:\Data\Perso\algo-trading-intraday-research\n...
